# Do these documents describe the same person?

**What this notebook does:** takes several unrelated PDFs — a bank statement,
a payslip, an invoice — pulls the identity out of each one, and decides whether
they refer to the same person. Then it signs that decision.

**Who it's for:** anyone new to this. Every step prints what it did and why.

**Runtime:** under a minute, offline.

---

> ## ⚠️ Read this before running
>
> This notebook processes **real personal documents**. It is designed so that
> nothing sensitive is printed by default — output is masked, and the raw
> values stay in memory rather than on screen.
>
> Two rules:
>
> 1. **Set `REVEAL = True` only on your own machine, on your own documents.**
> 2. **Clear all outputs before committing or sharing this notebook.**
>    `Kernel → Restart & Clear Output`, or
>    `jupyter nbconvert --clear-output --inplace 02_same_person_across_documents.ipynb`
>
> The whole point of arche is that this data deserves care. A demo that leaks
> it would be arguing against itself.
>
> **`data/docs/` is gitignored.** The documents behind the saved outputs are
> personal and are not distributed. Drop two or three of your own documents
> about the same person in there — a bank statement, a payslip, an invoice —
> and re-run. The saved outputs show what a real set produced, fully masked.

---

## The problem

You have three documents from three organisations. Nobody agreed on a format:

| Document | Who wrote it | What it calls the person |
|---|---|---|
| Bank statement | A UK bank | Full legal name, including a middle name |
| Payslip | An employer's payroll system | Short name, address on one line |
| Invoice | A US software vendor | Short name, address in a different order, plus an email |

No shared customer number. No national ID. Three spellings of one address, in
three different field orders. This is what "messy, multilingual, real-world
data" actually looks like in an inbox — and deciding whether it is one person
or three is the entity resolution problem in miniature.

## Step 0 — Setup

`REVEAL` controls whether real values are printed. Leave it `False` unless you
are working locally on documents you own.

In [1]:
import warnings, glob, collections, json
warnings.filterwarnings("ignore")

REVEAL = False          # <-- keep False when sharing this notebook

def show(value, keep=4):
    """Mask a value unless REVEAL is on. Keeps enough to be recognisable."""
    if REVEAL:
        return value
    s = str(value)
    if len(s) <= keep:
        return "*" * len(s)
    return s[:keep] + "*" * (len(s) - keep)

DOCS = sorted(glob.glob("../../data/docs/*.pdf"))

if not DOCS:
    raise SystemExit(
        "No PDFs found in data/docs/.\n\n"
        "That directory is gitignored — the documents used to produce the saved\n"
        "outputs below are personal and are not distributed with the repository.\n\n"
        "Drop two or three of YOUR OWN documents about the same person into\n"
        "data/docs/ and re-run. Anything works: a bank statement, a payslip, an\n"
        "invoice, a utility bill. The point is that they come from different\n"
        "organisations and disagree about formatting.\n\n"
        "The saved outputs below show what the notebook produced on a real set,\n"
        "with every value masked."
    )

print(f"found {len(DOCS)} documents")
for d in DOCS:
    print("   ", d.split("/")[-1].split("\\")[-1])

found 3 documents
    Emailing Monzo_bank_statement_2025-12-01-2026-02-28_2231.pdf
    Invoice-PEDHCF-00012.pdf
    Paystatement_2025-12-23T00_00_00.pdf


## Step 1 — Get text out of the PDFs

arche has a document substrate (`Pipeline.process_file`) that handles PDF,
DOCX, PPTX and XLSX — but it needs the `[doc]` extra, which pulls in docling
and is a heavy install. This notebook uses PyMuPDF directly so it stays light.

Either path ends up at the same place: a string of text you can run detection
over.

In [2]:
import fitz   # PyMuPDF

def read_pdf(path):
    doc = fitz.open(path)
    return "\n".join(page.get_text() for page in doc)

texts = {}
for path in DOCS:
    name = path.replace("\\", "/").split("/")[-1]
    texts[name] = read_pdf(path)
    print(f"{name[:46]:48} {len(texts[name]):>6,} characters")

Emailing Monzo_bank_statement_2025-12-01-2026-   16,520 characters
Invoice-PEDHCF-00012.pdf                          1,027 characters
Paystatement_2025-12-23T00_00_00.pdf              1,242 characters


## Step 2 — Detect the identifying data

`Pipeline` finds identifying data and applies the policy of a jurisdiction.
Which jurisdiction you pick is not cosmetic — it decides which patterns are
even looked for.

These are UK documents, so let's see what happens if we get that wrong.

In [3]:
from arche import Pipeline

sample = list(texts.values())[0]

for jurisdiction in ["NG", "GDPR"]:
    result = Pipeline(jurisdiction=jurisdiction).process(sample)
    counts = collections.Counter(d.category for d in result.detections)
    print(f"{jurisdiction:6} {dict(counts)}")

NG     {'PII-2-TIN': 36, 'PII-4-ADDRESS': 13, 'PII-3-PHONE': 1}
GDPR   {'PII-4-ADDRESS': 13, 'PII-3-PHONE': 1}


**That difference is the lesson.** Running a UK bank statement through the
Nigerian pack reports dozens of `PII-2-TIN` hits — Nigerian Tax Identification
Numbers that are really just transaction reference numbers that happen to match
the shape.

Switch to `GDPR` and they vanish, because the GDPR pack doesn't look for
Nigerian TINs.

A detector without a jurisdiction is a regex with opinions. The statute pack is
what makes a detection mean something — and what stops it meaning something it
shouldn't.

## Step 3 — An honest limitation you need to know about

Before trusting any of this, check what the detector *missed*. This is the
single most important habit when working with redaction.

In [4]:
probe = "Contact Amara Nwosu at amara@example.com about invoice ACME-00012."
r = Pipeline(jurisdiction="GDPR").process(probe)

print("input     :", probe)
print("detections:", [(d.category, d.text) for d in r.detections])
print("redacted  :", r.redacted_text)
print()
from arche.detect.emails import detect_emails
print("but detect_emails() alone finds:")
for d in detect_emails(probe):
    print("   ", d.category, "->", d.text)

input     : Contact Amara Nwosu at amara@example.com about invoice ACME-00012.
detections: []
redacted  : Contact Amara Nwosu at amara@example.com about invoice ACME-00012.

but detect_emails() alone finds:
    PII-3-EMAIL -> amara@example.com


**The redacted text is the input, unchanged.** `Pipeline` returned zero
detections: it missed the name *and* the email, and handed back text it had
done nothing to.

Two separate causes, both worth understanding:

1. **`detect_emails` is not in `Pipeline`'s default detector chain** in
   v0.3.0a1. It works perfectly on its own — you have to add it. This is a
   known issue, documented in the changelog.
2. **Name detection without a model** falls back to lexicons and rules. The
   African name lexicon is arche's strongest asset and it has no reason to
   contain a British-inflected name. Install `arche-core[detect]` for
   GLiNER-backed name detection.

**Never assume `redacted_text` is safe without checking what was detected.**
Print the detections. If the list is empty, nothing was redacted. That habit
would have caught this in seconds.

## Step 4 — Getting the fields out (and why this is the hard part)

We need one identity record per document. The obvious approach is a layout
heuristic: find the postcode, take the lines above it as the address, and take
the first name-shaped line as the person.

Let's watch that fail, because the failure is the most useful thing in this
notebook.

In [5]:
import re

ORG    = re.compile(r"\b(ltd|limited|inc|llc|plc|gmbh|corp|bank|company)\b", re.I)
PERSON = re.compile(r"^[A-Z][a-z]+(?: [A-Z][a-z\-\u2019\x27]+){1,3}$")
UK_PC  = re.compile(r"\b([A-Z]{1,2}\d{1,2}[A-Z]?\s+\d[A-Z]{2})\b")

def naive_name(text):
    for line in [l.strip() for l in text.split("\n") if l.strip()][:25]:
        if PERSON.match(line) and not ORG.search(line):
            return line
    return ""

for doc, text in texts.items():
    print(f"{doc[:44]:46} -> {show(naive_name(text))}")

Emailing Monzo_bank_statement_2025-12-01-202   -> Denn********************
Invoice-PEDHCF-00012.pdf                       -> Unit*********
Paystatement_2025-12-23T00_00_00.pdf           -> Pay *******


Three documents, three different answers — and at most one of them is the
person. `United States` and `Pay Summary` are shaped exactly like a name:
capitalised words, no digits, no company suffix. No amount of regex tuning
fixes this, because the pattern genuinely cannot distinguish a person from a
country by shape alone.

**This is why arche has a declaration layer.** In production you write a YAML
file naming *your* fields, and a model fills it in — see
`how-to/declare-your-schema.md` and `how-to/bring-your-own-llm.md`. The model
reads layout the way a person does; the declaration constrains what it is
allowed to return; hallucinated fields become violations rather than values.

For this notebook we use a trick that needs no model and no hardcoding.

### Finding the identity without knowing it in advance

Three documents from three organisations. Whatever they have **in common** is
the person. So instead of guessing which line is a name, intersect the
documents and let the shared lines identify themselves.

In [6]:
import collections

line_counts = collections.Counter()
for text in texts.values():
    for line in {l.strip() for l in text.split("\n") if l.strip()}:
        if 2 <= len(line.split()) <= 8:
            line_counts[line] += 1

shared = [line for line, n in line_counts.items() if n >= 2]
print(f"lines appearing in 2 or more documents ({len(shared)}):")
for line in shared:
    print("   ", show(line))

lines appearing in 2 or more documents (4):
    3 Ma**********
    Unit**********
    B16 ***
    Denn**********


That short list is the identity: a name, a street, a postcode, a country.
Nothing was hardcoded — the documents told us who they were about by agreeing
with each other.

Now we build one record per document, keeping each document's *own* spelling so
the comparison has something real to work on.

In [7]:
PERSON_LINE = [s for s in shared if PERSON.match(s) and not ORG.search(s)
               and not UK_PC.search(s)]
EMAIL = re.compile(r"\b([\w.\-]+@[\w.\-]+\.\w+)\b")

records = {}
for doc, text in texts.items():
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    rec = {}

    # the person's name as THIS document spells it
    for line in lines:
        if any(line.startswith(p.split()[0]) for p in PERSON_LINE) and not ORG.search(line):
            rec["name"] = line
            break

    m = EMAIL.search(text)
    if m:
        rec["email"] = m.group(1)

    for i, line in enumerate(lines):
        pc = UK_PC.search(line)
        if pc:
            rec["postcode"] = pc.group(1)
            block = [b for b in lines[max(0, i - 4):i] if b != rec.get("name")]
            rec["address"] = " ".join(block)
            break

    records[doc] = {k: v for k, v in rec.items() if v}

for doc, rec in records.items():
    print(f"--- {doc[:44]}")
    for k, v in rec.items():
        print(f"    {k:10} {show(v)}")

--- Emailing Monzo_bank_statement_2025-12-01-202
    name       Denn********************
    postcode   B16 ***
    address    Apar********************************************
--- Invoice-PEDHCF-00012.pdf
    name       Unit*********
    email      deni***************
    postcode   B16 ***
    address    Bill**********************************************************
--- Paystatement_2025-12-23T00_00_00.pdf
    name       Denn**********
    postcode   B16 ***
    address    Viat***********************************************


Notice the variation, even masked:

- One document carries a **middle name** the others drop.
- The address appears in a **different field order** in each — the same
  building written three ways.
- Only one document has an email.

A human glances at these and says "obviously the same person". A string
comparison says they're different. That gap is the entire problem.

## Step 5 — Compare them

`resolve.pairwise` scores a pair and returns a decision. It is the same engine
the facility notebook used — what changes is the *comparator*, because a person
is not a place.

It takes `Reference` objects rather than raw dicts. A `Reference` is arche's
canonical form: field names mapped to roles, values normalised, so that
`"SW1A 1AA"` and `"sw1a 1aa"` are the same postcode before anything is scored.

In [8]:
from arche.canonical import Reference
from arche import resolve
from itertools import combinations

refs = {doc: Reference.from_record(rec) for doc, rec in records.items()}

names = list(refs)
rows = []
for a, b in combinations(names, 2):
    d = resolve.pairwise(refs[a], refs[b], entity="person")
    rows.append((a, b, d))
    print(f"{a[:26]:28} vs {b[:26]:28} -> {d.identity:12} {d.score:.4f}")

Emailing Monzo_bank_statem   vs Invoice-PEDHCF-00012.pdf     -> review       0.9877
Emailing Monzo_bank_statem   vs Paystatement_2025-12-23T00   -> review       0.9974
Invoice-PEDHCF-00012.pdf     vs Paystatement_2025-12-23T00   -> review       0.9836


## Step 6 — The answer table

This is what you came for: do these documents describe the same person?

In [9]:
print(f"{'Document A':30} {'Document B':30} {'verdict':14} {'score':>7}  evidence")
print("-" * 118)
for a, b, d in rows:
    ev = ", ".join(f"{k}={v:.2f}" for k, v in sorted(d.factors.items())
                   if isinstance(v, (int, float)))
    print(f"{a[:28]:30} {b[:28]:30} {d.identity:14} {d.score:>7.4f}  {ev[:44]}")

Document A                     Document B                     verdict          score  evidence
----------------------------------------------------------------------------------------------------------------------
Emailing Monzo_bank_statemen   Invoice-PEDHCF-00012.pdf       review          0.9877  address=1.00, name=0.58, name_tf=0.00
Emailing Monzo_bank_statemen   Paystatement_2025-12-23T00_0   review          0.9974  address=1.00, name=0.80, name_tf=0.64
Invoice-PEDHCF-00012.pdf       Paystatement_2025-12-23T00_0   review          0.9836  address=1.00, name=0.54, name_tf=0.00


### How to read a verdict

| Verdict | Meaning |
|---|---|
| `same_entity` | The evidence is distinctive enough to merge these records |
| `review` | Plausible, but not distinctive enough. **A human decides.** |
| `different` | The evidence actively contradicts a merge |

`review` is not a failure. Consider what the alternative costs: if these were
customer records at a bank, a wrong merge means one person seeing another
person's transactions. Under GDPR that is a reportable data breach. The system
that says *"I'm not sure, look at this one"* is the one you want.

## Step 7 — Why the score alone won't tell you

A common instinct is to skip the verdict and threshold on the score. Here is
why that breaks.

In [10]:
a = Reference.from_record({"name": "Fatima Abdullahi", "national_id": "12345678901"})
b = Reference.from_record({"name": "Fatuma Abdulahi",  "national_id": "12345678901"})
d = resolve.pairwise(a, b, entity="person")
print(f"identical national ID, near-identical name -> {d.identity} at {d.score}")
print()
print("factors:", json.dumps(d.factors, indent=2, default=str))

identical national ID, near-identical name -> same_entity at 1.0

factors: {
  "name": 0.9053,
  "national_id": 1.0,
  "name_tf": 0.0
}


**0.9999, and still `review`.**

Sharing an exact national ID is powerful evidence. But the gate asks a
different question: is the evidence *distinctive*? A shared ID in a dataset
where IDs are sometimes recycled, mistyped, or shared between family members is
not automatically proof of identity.

If you had thresholded at 0.95 you would have merged them. arche makes you look
at it. **That is the design, not a rough edge** — and it is why the score is
reported alongside the verdict rather than instead of it.

## Step 8 — Sign the decision

The last step is what makes this usable in a regulated setting. A decision is
only worth as much as your ability to defend it later.

In [11]:
from arche.attest import attest, verify_attestation
from arche.sign import generate_keypair

ISSUER_KEY = b"replace-with-a-real-32-byte-secret-key!!"   # >= 32 bytes
keypair = generate_keypair()

a, b, _ = rows[0]
decision = resolve.pairwise(refs[a], refs[b], entity="person",
                            issuer_key=ISSUER_KEY)
signed = attest(decision, keypair, mode="jws")

check = verify_attestation(signed.compact, public_key=keypair.public_key)
print("valid       :", check.valid)
print("trusted     :", check.trusted, " <- key came from somewhere we control")
print("reproducible:", check.reproducible)
print("decision_id :", check.decision_id[:44], "...")

valid       : True
trusted     : True  <- key came from somewhere we control
reproducible: True
decision_id : dec:hmac-sha256:73d78d663470096f34309612aaa9 ...


Three fields, three different questions:

- **`valid`** — does the signature match the key?
- **`trusted`** — did that key come from somewhere *we* control? Verifying
  without pinning a key proves a token is internally consistent, not who made
  it. Always check `trusted`, not just `valid`.
- **`reproducible`** — can this decision be replayed from its evidence? Here
  it's `True` because the engine did the deciding. Had an LLM extracted the
  fields, it would be `False`, and the signature would say so.

`decision_id` is a content hash over the evidence *and* the exact
representation that produced it. Same inputs tomorrow, same id.

## Step 9 — Sharing results without leaking the person

If any of this leaves your machine, the values must not go with it.

In [12]:
safe = []
for a, b, d in rows:
    safe.append({
        "doc_a": a,
        "doc_b": b,
        "verdict": d.identity,
        "score": round(d.score, 4),
        "decision_id": d.decision_id,
    })

print(json.dumps(safe, indent=2))
print()
print("No name, no address, no email — but the decision is still fully auditable")
print("because decision_id pins the evidence that produced it.")

[
  {
    "doc_a": "Emailing Monzo_bank_statement_2025-12-01-2026-02-28_2231.pdf",
    "doc_b": "Invoice-PEDHCF-00012.pdf",
    "verdict": "review",
    "score": 0.9877,
    "decision_id": "dec:sha256:b3f58e270d85ec2b46804820973d842af94cff31c486eef30633cb74b3490eb3"
  },
  {
    "doc_a": "Emailing Monzo_bank_statement_2025-12-01-2026-02-28_2231.pdf",
    "doc_b": "Paystatement_2025-12-23T00_00_00.pdf",
    "verdict": "review",
    "score": 0.9974,
    "decision_id": "dec:sha256:3f37371ab8b4e11215fed5d627cc7253f9ec7be2ec7cb38ade43a8fe36a8edeb"
  },
  {
    "doc_a": "Invoice-PEDHCF-00012.pdf",
    "doc_b": "Paystatement_2025-12-23T00_00_00.pdf",
    "verdict": "review",
    "score": 0.9836,
    "decision_id": "dec:sha256:e4029389b3df0e88a88b70d85006e4da726e352e2cef9016cde02bb81f4c41ff"
  }
]

No name, no address, no email — but the decision is still fully auditable
because decision_id pins the evidence that produced it.


## What you just did

1. Read three real PDFs written by three organisations that never coordinated
2. Saw that the wrong jurisdiction invents PII that isn't there
3. **Checked what the detector missed**, and found the redaction had done
   nothing — the habit that matters most
4. Built one identity record per document
5. Compared them with a person-calibrated comparator
6. Got a table of verdicts, with `review` where a human is genuinely needed
7. Saw why a score alone is not a decision
8. Signed a decision so it can be defended later
9. Exported results that carry no personal data at all

## Before you close this notebook

```
Kernel → Restart & Clear Output
```

Or from a terminal:

```bash
jupyter nbconvert --clear-output --inplace 02_same_person_across_documents.ipynb
```

## Try it yourself

- Set `REVEAL = True` locally and re-read Step 4 — the address variations are
  more striking unmasked
- Add `detect_emails` to the pipeline and see the email get caught
- Install `arche-core[detect]` and re-run Step 3 with GLiNER name detection
- Point `DOCS` at your own documents and see what survives

## Next

- **Declare your schema** — replace the regex in Step 4 with a YAML declaration
- **Bring your own LLM** — have a model fill that declaration, and let the
  engine grade it